# Lezione 9 — Loss function: come si misura l'errore

**Come si usa questo notebook.** Come sempre. Prerequisito: Lezione 8.
Chiude la Fase 0: alla fine il progetto avrà il suo **primo classificatore
addestrato**, costruito interamente a mano.

**Cosa saprai fare alla fine:** scegliere la funzione di perdita giusta per
il problema, e capire perché "quale errore minimizzi" determina *cosa* il
modello impara davvero.

## Parte 1 — Teoria: la loss è il contratto col modello

La discesa del gradiente (Lezione 8) minimizza *quello che le dici di
minimizzare* — né più né meno. La **loss function** è quindi il contratto:
definisce cosa significa "sbagliare" e di conseguenza cosa il modello
ottimizza. Sceglierla male = addestrare il modello a fare la cosa sbagliata,
con precisione.

**Per la regressione** (predire un numero): lo scarto quadratico medio
(**MSE**), che hai già usato. Il quadrato penalizza molto gli errori grandi
e poco i piccoli; per questo è sensibile agli outlier (eco della Lezione 2:
un outlier non gestito *domina* la loss).

**Per la classificazione** serve altro. I punteggi della Lezione 7 vanno
prima trasformati in **probabilità** con la **softmax**: esponenzia ogni
punteggio e normalizza, così escono numeri positivi che sommano a 1. Poi la
**cross-entropy** misura l'errore così:

> guarda solo la probabilità assegnata alla **classe giusta**, e penalizza
> con `-log(p)`: se p=0.9 la penalità è piccola (0.10); se p=0.01 la
> penalità è enorme (4.6).

Il `-log` è il cuore: **essere sicuri e sbagliati costa tantissimo**,
essere incerti costa poco. È esattamente l'incentivo giusto per un
classificatore, e ha anche gradienti "puliti" che rendono la discesa
efficiente — per questo è lo standard universale.

In [ ]:
import numpy as np

def softmax(punteggi):
    e = np.exp(punteggi - punteggi.max(axis=-1, keepdims=True))  # stabile numericamente
    return e / e.sum(axis=-1, keepdims=True)

punteggi = np.array([2.0, 0.5, -1.0, 0.0])   # 4 classi, come nel progetto
p = softmax(punteggi)
print('probabilita:', p.round(3), '  somma:', p.sum())

print('\nPenalita -log(p) se la classe giusta ha probabilita p:')
for prob in [0.9, 0.5, 0.1, 0.01]:
    print(f'  p={prob:4}: {-np.log(prob):5.2f}')

## Parte 2 — Esercizio guidato

Un modello deve dire se una memoria è `preference` (sì/no). Due candidati
danno queste probabilità alla classe giusta su 4 esempi:

- modello A: `[0.6, 0.6, 0.6, 0.45]` — sempre prudente, un errore di misura;
- modello B: `[0.99, 0.99, 0.99, 0.03]` — quasi perfetto tre volte,
  **sicuro e sbagliato** una volta.

Il tuo compito: calcola l'**accuratezza** (soglia 0.5) e la **cross-entropy
media** (`-log(p)` medio) di entrambi. Prima di eseguire, scommetti: le due
metriche daranno lo stesso verdetto?

**Prova tu:**

In [ ]:
p_A = np.array([0.6, 0.6, 0.6, 0.45])
p_B = np.array([0.99, 0.99, 0.99, 0.03])

# Scrivi qui: accuratezza (p > 0.5) e cross-entropy media di A e B

pass

### Soluzione spiegata

Per l'accuratezza i due modelli sono **identici**: 3 su 4 entrambi. La
cross-entropy invece li separa nettamente: il singolo `-log(0.03)` di B
pesa più di tutti i suoi successi, mentre gli errori "prudenti" di A
costano poco. Le due metriche misurano cose diverse: l'accuratezza conta le
decisioni, la cross-entropy misura la **qualità delle probabilità** — e in
produzione (soglie, ranking, decisioni costose) le probabilità contano.

In [ ]:
for nome, p in [('A', p_A), ('B', p_B)]:
    accuratezza = (p > 0.5).mean()
    ce = -np.log(p).mean()
    print(f'modello {nome}: accuratezza {accuratezza:.0%}   cross-entropy {ce:.2f}')

## Parte 3 — Il progetto: Memory AI Lab, passo 9 (fine Fase 0)

Tutti i pezzi delle ultime quattro lezioni si incastrano adesso:

- la matrice `X` e i punteggi `X @ W` (Lezioni 6-7);
- softmax + cross-entropy come loss (oggi);
- la discesa del gradiente per imparare `W` (Lezione 8) — con un fatto
  elegante: per softmax + cross-entropy, il gradiente dei punteggi è
  semplicemente `probabilita - verita` (la chain rule fa collassare tutto
  in questa differenza).

Risultato: una **softmax regression** — il classificatore lineare completo,
l'antenato diretto di ogni rete neurale. La Fase 2 aggiungerà strati e
non-linearità; il ciclo di training resterà *identico* a questo.

In [ ]:
import pandas as pd
from pathlib import Path

processed = Path('..') / 'datasets' / 'processed'
insiemi = {n: pd.read_csv(processed / f'memory_features_{n}.csv') for n in ['train', 'val']}
X = {n: f.drop(columns='target').to_numpy() for n, f in insiemi.items()}
y = {n: f['target'].to_numpy() for n, f in insiemi.items()}

n_classi = 4
Y_onehot = np.eye(n_classi)[y['train']]          # verita' in formato one-hot (Lezione 5!)
W = np.zeros((X['train'].shape[1], n_classi))    # partenza neutra: zero
b = np.zeros(n_classi)
passo = 0.5

for epoca in range(300):
    p = softmax(X['train'] @ W + b)              # forward: punteggi -> probabilita'
    grad_punteggi = (p - Y_onehot) / len(Y_onehot)   # il gradiente elegante
    W -= passo * X['train'].T @ grad_punteggi    # chain rule fino ai pesi
    b -= passo * grad_punteggi.sum(axis=0)
    if epoca % 100 == 0:
        loss = -np.log(p[np.arange(len(p)), y['train']] + 1e-12).mean()
        print(f'epoca {epoca:3d}: cross-entropy {loss:.3f}')

In [ ]:
def accuratezza(nome):
    predizioni = (X[nome] @ W + b).argmax(axis=1)
    return (predizioni == y[nome]).mean()

frequente = np.bincount(y['train']).max() / len(y['train'])
print(f'accuratezza train      : {accuratezza("train"):.0%}')
print(f'accuratezza validation : {accuratezza("val"):.0%}')
print(f'baseline classe frequente: {frequente:.0%}   caso puro: 25%')

np.save(processed / 'softmax_baseline_W.npy', W)
np.save(processed / 'softmax_baseline_b.npy', b)
print('\nParametri salvati: la Fase 2 dovra battere questa baseline.')

Leggi i numeri con occhio onesto (Lezione 4): se validation batte le
baseline, le sette feature povere (lunghezze, flag) contengono già segnale;
il divario tra train e validation ti dice quanto il modello memorizza.
Qualunque sia il risultato, ora è **la baseline ufficiale del progetto**,
salvata su disco: la prima rete Keras della Fase 2 dovrà batterla, e tu
saprai giudicare se ci riesce.

## Cosa hai imparato — e la Fase 0 è completa

- La loss è il **contratto**: il modello ottimizza esattamente quella.
- MSE per la regressione (attento agli outlier); **softmax +
  cross-entropy** per la classificazione.
- Il `-log(p)` punisce ferocemente la sicurezza sbagliata; accuratezza e
  cross-entropy possono dare giudizi opposti.
- Il ciclo completo di training — forward, loss, gradiente, passo — l'hai
  scritto **tutto a mano**. Keras automatizza *questo*, niente di più
  misterioso.

## Quiz

1. Perché per classificare non si usa l'MSE sulle probabilità?
2. Un modello dà probabilità 0.97 alla classe sbagliata. Quanto paga in
   cross-entropy per quell'esempio (ordine di grandezza)?
3. Nel training del progetto, W parte da zero eppure il modello impara:
   perché nella Fase 2, con le reti a più strati, la partenza da zero non
   funzionerà più? (Anticipazione: pensa a cosa rende diversi due neuroni.)

<details>
<summary><b>Apri le risposte</b></summary>

1. Si può, ma punisce poco gli errori sicuri (la differenza quadratica tra
   0.97 e 0 resta sotto 1) e produce gradienti deboli proprio dove serve
   correggere di più. La cross-entropy con -log ha l'incentivo e i
   gradienti giusti.
2. La classe giusta riceve al massimo 0.03, quindi -log(0.03) ≈ 3.5: più
   di trenta volte il costo di una previsione sicura corretta (~0.1).
3. Con un solo strato lineare la soluzione è convessa e la partenza è
   irrilevante. Con più strati, neuroni identici (tutti zero) ricevono
   gradienti identici e restano identici per sempre: serve rompere la
   simmetria con un'inizializzazione casuale — lo vedrai fare a Keras.
</details>

## Fonti

- Goodfellow, Bengio, Courville, *Deep Learning*, cap. 5-6 (loss, maximum
  likelihood, softmax): https://www.deeplearningbook.org/
- scikit-learn, *Log loss*:
  https://scikit-learn.org/stable/modules/model_evaluation.html#log-loss

## Prossimo passo del corso

Fase 0 completa: array, forme, gradienti, loss — e una baseline addestrata
a mano. Nella **Fase 2** arriva TensorFlow/Keras: la stessa identica
meccanica, con strati multipli, non-linearità e l'autodiff che calcola i
gradienti per te. Il progetto ha già pronto il dataset e l'asticella da
superare.